# Preprocessing Postprocessing

Reads results from `prep_convergence.json` written by the `Prep` class and produces analysis plots.  
**No DFT calculations are run here.**

Sections:
0. Pseudopotential notes
1. Lattice constant optimization
2. Cutoff energy (ecut) convergence
3. k-point convergence

In [ ]:
import json
import os
import sys
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
from preprocessing import plot_lattice_constant, plot_ecut_convergence, plot_kpoint_convergence

%matplotlib inline

In [ ]:
output_path = "prep_convergence.json"

with open(output_path) as f:
    out = json.load(f)

steps = {s["step"]: s for s in out["steps"] if s.get("status") == "completed"}
print("Completed steps:", list(steps.keys()))

---
## 0. Pseudopotentials

The choice of pseudopotential affects accuracy and transferability. Key considerations:

| Type | Description | Use when |
|------|-------------|----------|
| **NC (Norm-Conserving)** | Strictly enforces norm inside cutoff radius. Higher plane-wave cutoff needed. | High accuracy, small systems |
| **US (Ultrasoft)** | Relaxes norm constraint → lower cutoff. `ecutrho` ≈ 8–12× `ecutwfc`. | Large systems, transition metals |
| **PAW (Projector Augmented Wave)** | All-electron quality near nucleus, smooth valence. `ecutrho` ≈ 4× `ecutwfc`. | Production QE runs, recommended for most cases |

**Recommended libraries:**
- [Quantum ESPRESSO official table](https://www.quantum-espresso.org/pseudopotentials) — SSSP efficiency/precision sets
- [Materials Cloud SSSP](https://www.materialscloud.org/discover/sssp/table/efficiency) — benchmarked, recommended starting point
- [PseudoDojo](http://www.pseudo-dojo.org) — NC and PAW, supports ONCVPSP
- [GBRV](https://www.physics.rutgers.edu/gbrv/) — US pseudopotentials optimized for BEEF-vdW
- [BLYP/HGH](https://pseudopotentials.quantum-espresso.org/legacy_tables/hartwigesen-goedecker-hutter-pp)
- [SCAN TM](https://yaoyi92.github.io/scan-tm-pseudopotentials.html) — for SCAN functional

**Tips:**
- For heavy elements with spin-orbit coupling → use fully relativistic pseudopotentials.
- Always verify your pseudopotential reproduces known bulk properties (lattice constant, bulk modulus) before production runs.
- For BEEF-vdW, GBRV US pseudopotentials are a common, well-tested choice.

In [ ]:
pseudos = out["user_arguments"]["software_kwargs"]["pseudopotentials"]
for el, pp in pseudos.items():
    print(f"  {el:4s}: {pp}")

---
## 1. Lattice Constant Optimization

The `Prep` class scans total energy vs. lattice constant *a* and fits a quadratic to find the minimum.  
Scan data (`scan_avals_A`, `scan_Evals_eV`) is saved when running `calculate_alloy_lattice_parameters`.  
For `calculate_lattice_parameters` (pure metal), only the final value is stored.

In [ ]:
lout = (steps.get("calculate_alloy_lattice_parameters") or steps["calculate_lattice_parameters"])["outputs"]
a_opt = plot_lattice_constant(lout)

---
## 2. Cutoff Energy (ecut) Convergence

Energy is plotted vs. `ecutwfc` at the finest k-point mesh tested.  
The relative panel (right) shows |ΔE| with respect to the highest ecut — a converged calculation should show |ΔE| below your target threshold (commonly < 1 meV/atom).

In [ ]:
results = (steps.get("run_convergence_adsorbate") or steps["run_convergence"])["outputs"]["results"]
ecuts_all, kpts_all = plot_ecut_convergence(results)

---
## 3. k-point Convergence

Energy is plotted vs. k-mesh at the finest ecut tested.  
A denser k-mesh improves accuracy but increases cost. Choose the coarsest mesh where |ΔE| is below your threshold.

In [ ]:
plot_kpoint_convergence(results)

---
## Summary

In [ ]:
print("=" * 50)
print("PREPROCESSING SUMMARY")
print("=" * 50)
print(f"Lattice constant : {a_opt:.6f} Å")
print(f"ecut tested      : {ecuts_all} Ry")
print(f"kpts tested      : {kpts_all}")
print("=" * 50)